## Import Libraries

In [1]:
import numpy as np

import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

## Prepare train dataset

In [2]:
# Transform to turn into tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081,)) # mean y std
])

In [3]:
# Load the dataset and normalize it
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

## Prepare test dataset

In [4]:
transform_test = transforms.ToTensor()

In [5]:
# Load the test dataset
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=False,
    transform=transform_test
)

## Data Loaders

In [6]:
# DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=len(test_dataset[0][0][0])
)

## Feed Forward Network 

In [7]:
# All variables
%whos

Variable         Type          Data/Info
----------------------------------------
DataLoader       type          <class 'torch.utils.data.dataloader.DataLoader'>
F                module        <module 'torch.nn.functio<...>/torch/nn/functional.py'>
datasets         module        <module 'torchvision.data<...>on/datasets/__init__.py'>
nn               module        <module 'torch.nn' from '<...>es/torch/nn/__init__.py'>
np               module        <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
plt              module        <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
test_dataset     MNIST         Dataset MNIST\n    Number<...>rm\nTransform: ToTensor()
test_loader      DataLoader    <torch.utils.data.dataloa<...>object at 0x7fca9d825cd0>
torch            module        <module 'torch' from '/ho<...>kages/torch/__init__.py'>
train_dataset    MNIST         Dataset MNIST\n    Number<...>=(0.3081,))\n           )
train_loader     DataLoader    <torch.utils.data.datalo

In [8]:
def createTheMNISTNet():

    class mnistNet(nn.Module):
        def __init__(self):
            super().__init__()

            # input layer
            self.input = nn.Linear(784,64) # 784 pixels

            # hidden layer
            self.fc1 = nn.Linear(64,32)
            self.fc2 = nn.Linear(32,32)

            # output layer
            self.output = nn.Linear(32,10)

        def forward(self,x):
            x = x.view(x.shape[0], -1)
            
            x = F.relu( self.input(x) )
            x = F.relu( self.fc1(x) )
            x = F.relu( self.fc2(x) )
            return torch.log_softmax( self.output(x), dim=1 )

    net = mnistNet()

    lossfun = nn.NLLLoss()

    optimizer = torch.optim.SGD(net.parameters(), lr=0.1) # try with Adam later

    return net, lossfun, optimizer

In [9]:
net, lossfun, optimizer = createTheMNISTNet()

X,y = next(iter(train_loader))
yHat = net(X)

print(yHat)

loss = lossfun(yHat, y)
print(' ')
print('Loss:')
print(loss)

tensor([[-2.0910, -2.2694, -2.2693, -2.2525, -2.4570, -2.3983, -2.3711, -2.3114,
         -2.2193, -2.4443],
        [-2.0955, -2.2776, -2.2694, -2.2359, -2.4448, -2.3947, -2.3845, -2.3092,
         -2.1991, -2.4784],
        [-2.1053, -2.2259, -2.2994, -2.2504, -2.4934, -2.4239, -2.3558, -2.3010,
         -2.1892, -2.4492],
        [-2.1116, -2.2321, -2.3059, -2.2597, -2.4834, -2.4015, -2.3469, -2.2880,
         -2.2217, -2.4302],
        [-2.0897, -2.2548, -2.2619, -2.2607, -2.5303, -2.4219, -2.3638, -2.2805,
         -2.1842, -2.4569],
        [-2.0701, -2.2775, -2.2895, -2.2542, -2.4433, -2.4142, -2.3700, -2.3241,
         -2.1960, -2.4524],
        [-2.0754, -2.2890, -2.2629, -2.2410, -2.4251, -2.4039, -2.3929, -2.3049,
         -2.2027, -2.4980],
        [-2.1065, -2.2310, -2.2779, -2.2658, -2.4394, -2.3951, -2.3779, -2.2930,
         -2.2238, -2.4723],
        [-2.0433, -2.2860, -2.2987, -2.2291, -2.4066, -2.4409, -2.3732, -2.3436,
         -2.2081, -2.4720],
        [-2.0854, -

## Training

In [12]:
def function2trainTheModel():

    numepochs = 60

    net, loss, optimizer = createTheMNISTNet()

    losses = torch.zeros(numepochs)
    trainAcc = []
    testAcc = []

    for epochi in range(numepochs):

        # ===== Train =====
        batchAcc = []
        batchLoss = []
        for X,y in train_loader:
            
            # forward
            yHat = net(X)
            loss = lossfun(yHat, y)

            # back
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss
            batchLoss.append( loss.item() )

            # accuracy
            matches = torch.argmax(yHat, axis=1) == y # predictions
            matchesNumeric = matches.float() # to numeric
            accuracyPct = 100*torch.mean(matchesNumeric) # pct of mean
            batchAcc.append( accuracyPct ) # add

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        # ===== Test =====
        X,y = next(iter(test_loader))
        yHat = net(X)

        testAcc.append( 100*torch.mean( (torch.argmax(yHat, axis=1)==y).float() ) )

        print(f'Epoch: {epochi} \f trainAcc: {trainAcc[epochi]} \f testAcc: {testAcc[epochi]}')

    return trainAcc, testAcc, losses, net

In [13]:
trainAcc, testAcc, losses, net = function2trainTheModel()

Epoch: 0  trainAcc: 89.68000030517578  testAcc: 89.28571319580078
Epoch: 1  trainAcc: 96.1683349609375  testAcc: 89.28571319580078
Epoch: 2  trainAcc: 97.15333557128906  testAcc: 96.42857360839844


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16,5))

ax[0].plot(losses)
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_ylim([0,3])
ax[0].set_title('Model loss')

ax[1].plot(trainAcc, label='Train')
ax[1].plot(testAcc, label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_ylim([10,100])
ax[1].set_title(f'Final model test accuracy: {testAcc[-1]:.2f}%')
ax[1].legend()